# Lab 25-12 · Shmoo Plots, Bug Discovery and Fix Decisions

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import postsi as p

## Part 1 · Trace window

*Predict* the storage for 256 signals over 50 µs at 2 GHz.

In [2]:
depth = int(50e-6 * 2e9)
bits, window = p.trace(256, depth, 2e9)
print(f"depth {depth}, storage {bits/8/2**20:.2f} MiB, window {window*1e6:.1f} us")

depth 100000, storage 3.05 MiB, window 50.0 us


## Part 2 · Shmoo

*Predict* how close the fitted K comes to the true one.

In [3]:
k_true = p.fit_k(3.0, 0.75)
volts = np.linspace(0.55, 0.95, 41)
freqs = np.linspace(1.5, 4.5, 61)
grid = p.shmoo(k_true, volts, freqs)
k_fit = p.fit_shmoo_k(grid, volts, freqs)
print(f"true K {k_true:.3f}, fitted {k_fit:.3f} ({100*(k_fit/k_true-1):+.1f} %)")
print(f"Vmin for 2.0 GHz (true model): {p.vmin(2.0, k_true):.3f} V")
plt.figure(figsize=(6, 4))
vv, ff = np.meshgrid(volts, freqs)
plt.scatter(vv[grid], ff[grid], s=4, c="tab:green", label="pass")
plt.scatter(vv[~grid], ff[~grid], s=4, c="tab:red", label="fail")
plt.plot(volts, [p.fmax(v, k_fit) for v in volts], "k-", label="fitted boundary")
plt.xlabel("supply (V)"); plt.ylabel("frequency (GHz)"); plt.legend(); plt.grid(True)
plt.show()

true K 6.353, fitted 6.307 (-0.7 %)
Vmin for 2.0 GHz (true model): 0.565 V


## Part 3 · Bug discovery

*Predict* the fitted total bug count from a noisy weekly log.

In [4]:
rng = np.random.default_rng(7)
weeks = np.arange(1, 13)
truth = 120 * (1 - np.exp(-0.173 * weeks))
log = np.maximum.accumulate(np.round(truth + rng.normal(0, 3, weeks.size)))
a, b = p.go_fit(weeks, log)
print(f"fitted a = {a:.0f}, b = {b:.3f} per week")
t = np.linspace(0, 30, 200)
plt.figure()
plt.plot(weeks, log, "o", label="bug log")
plt.plot(t, a * (1 - np.exp(-b * t)), label="fit")
plt.axhline(a, ls=":", color="k")
plt.xlabel("weeks"); plt.ylabel("cumulative bugs"); plt.legend(); plt.grid(True)
plt.show()

fitted a = 124, b = 0.161 per week


## Part 4 · Decisions and statistics

*Predict* the break-even probability at $2 M per week.

In [5]:
print("decision at $5M/week:", p.decision(5.0))
print("decision at $2M/week:", p.decision(2.0))
lo, hi = p.wilson(16, 320)
print(f"Wilson interval for 16/320: {100*lo:.1f}% to {100*hi:.1f}%")

decision at $5M/week: (77.0, 125.0, 0.41600000000000004)
decision at $2M/week: (35.0, 65.0, 0.3384615384615385)
Wilson interval for 16/320: 3.1% to 8.0%
